# Reference Resolution Dev Notebook

In [31]:
import random
import torch
import io
import pyarrow as pa
import os
import copy
import pytorch_lightning as pl
from sacred import Experiment
from PIL import Image
from tqdm.auto import tqdm
import numpy as np
import skimage.io as skio
import matplotlib.pyplot as plt
from refer import REFER

from torch.optim import AdamW

from transformers import ElectraTokenizer

from refcoco_utils import get_bounded_subimage
from refcoco_utils import _config
from refcoco_utils import _loss_names

from meter.transforms import keys_to_transforms
from meter.config import ex
from meter.modules import METERTransformerSS
from meter.datamodules.multitask_datamodule import MTDataModule
from meter.datasets.base_dataset import BaseDataset

## RefCOCO Data

### Utility Functions

### Import and Explore Data

In [32]:
data_root = '/home/claytonfields/nlp/code/data/coco'  # contains refclef, refcoco, refcoco+, refcocog and images
dataset = 'refcoco' 
splitBy = 'unc'
refer = REFER(data_root, dataset, splitBy)

loading dataset refcoco into memory...
testing
creating index...
index created.
DONE (t=10.50s)


In [33]:
refer.IMAGE_DIR = '/home/claytonfields/nlp/code/data/coco/images/mscoco/train2014'

## METER Model

### Config

In [34]:
_config = copy.deepcopy(_config)
pl.seed_everything(_config["seed"])

dm = MTDataModule(_config, dist=False)
model = METERTransformerSS(_config)
exp_name = f'{_config["exp_name"]}'
os.makedirs(_config["log_dir"], exist_ok=True)
checkpoint_callback = pl.callbacks.ModelCheckpoint(
    save_top_k=1,
    verbose=True,
    monitor="val/the_metric",
    mode="max",
    save_last=True,
)
logger = pl.loggers.TensorBoardLogger(
    _config["log_dir"],
    name=f'{exp_name}_seed{_config["seed"]}_from_{_config["load_path"].split("/")[-1][:-5]}',
)

lr_callback = pl.callbacks.LearningRateMonitor(logging_interval="step")
callbacks = [checkpoint_callback, lr_callback]

num_gpus = (
    _config["num_gpus"]
    if isinstance(_config["num_gpus"], int)
    else len(_config["num_gpus"])
)

grad_steps = max(_config["batch_size"] // (
    _config["per_gpu_batchsize"] * num_gpus * _config["num_nodes"]
), 1)

max_steps = _config["max_steps"] if _config["max_steps"] is not None else None

trainer = pl.Trainer(
    gpus=_config["num_gpus"],
    num_nodes=_config["num_nodes"],
    precision=_config["precision"],
    benchmark=True,
    deterministic=True,
    max_epochs=_config["max_epoch"] if max_steps is None else 1000,
    max_steps=max_steps,
    callbacks=callbacks,
    logger=logger,
    #prepare_data_per_node=False,
    #replace_sampler_ddp=False,
    accumulate_grad_batches=grad_steps,
    log_every_n_steps=10,
    flush_logs_every_n_steps=10,
    resume_from_checkpoint=_config["resume_from"],
    weights_summary="top",
    fast_dev_run=_config["fast_dev_run"],
    val_check_interval=_config["val_check_interval"],
)

# if not _config["test_only"]:
#     trainer.fit(model, datamodule=dm)
# else:
#     trainer.test(model, datamodule=dm)

Global seed set to 0
Some weights of the model checkpoint at google/electra-small-discriminator were not used when initializing ElectraModel: ['discriminator_predictions.dense_prediction.bias', 'discriminator_predictions.dense_prediction.weight', 'discriminator_predictions.dense.bias', 'discriminator_predictions.dense.weight']
- This IS expected if you are initializing ElectraModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ElectraModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
GPU available: True, used: True
TPU available: False, using: 0 TPU cores


In [5]:
def get_sent_ids(refer):
    sent_ids = []
    for ref_id in refer.getRefIds():
        ref = refer.Refs[ref_id]
        for sent_id in ref['sent_ids']:
            sent_ids.append(sent_id)
    return sent_ids
get_sent_ids(refer)[-1]

142209

### Write Data Class

In [6]:
class RefcocoDataset(torch.utils.data.Dataset):

    def __init__(self, refer, tokenizer, max_bb = 75):
        self.tokenizer = tokenizer
        self.refer = refer
        self.max_bb = max_bb
#         self.targets = labels
        self.train_ids = refer.getRefIds(split='train')
#         self.max_len = max_len

    def __len__(self):
        return len(self.train_ids)

    def __getitem__(self, index):
        ref = refer.Refs[index]
        img_id = ref['image_id']
        ann_id = ref['ann_id']
        objs = refer.imgToAnns[img_id]
        obj_ids = [obj['id'] for obj in objs]
        
        sub_images = []
        for obj in objs:
            x_a = get_bounded_subimage(refer, img_id, obj['id'], xs=224,ys=224, show=False)
            if x_a is not None:
                sub_images.append(x_a)
        num_sub_images = len(sub_images)
        tokenized_sents = []
        text_masks = []
        text_labels = []
        text = []
        for sent in ref['sentences']:
            
            
            text_ids = tokenizer.encode(
                sent['sent'],
                padding="max_length",
                truncation=True,
                max_length=40,
                return_special_tokens_mask=True,
            )
            text_masks.append(torch.tensor([1 if text_ids[i]>0 else 0 for i,_ in enumerate(text_ids)]))
            text_labels.append(torch.tensor([-100 for i in range(40)]))
            text_ids = torch.tensor(text_ids)#.reshape(1,-1)
            tokenized_sents.append(text_ids)
            text.append(sent['sent'])
            
        ### TODO: Put all of the sub images in the infer dict with the coressponding sentence.
          
        return_dict = {
            'ann_id' : ann_id,
            'image' : sub_images,
            'obj_ids' : obj_ids,
            'text' : ref['sentences'],
            'text_ids' : tokenized_sents,
            'text_labels' : text_labels,
            'text_masks' : text_masks
        }


        return return_dict

## Ref Res with METER

In [7]:
optim = AdamW(model.parameters(), lr=1e-4)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# Ref Res with METER
tokenizer = ElectraTokenizer.from_pretrained('google/electra-small-discriminator')
BATCH_SIZE = 1


epochs = 1
# loader = dm.train_dataloader()
optim = AdamW(model.parameters(), lr=1e-4)
loss_fn = torch.nn.functional.cross_entropy
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [82]:
refer.sentToRef[0]

{'sent_ids': [0, 1, 2],
 'file_name': 'COCO_train2014_000000581857_16.jpg',
 'ann_id': 1719310,
 'ref_id': 0,
 'image_id': 581857,
 'split': 'train',
 'sentences': [{'tokens': ['the', 'lady', 'with', 'the', 'blue', 'shirt'],
   'raw': 'THE LADY WITH THE BLUE SHIRT',
   'sent_id': 0,
   'sent': 'the lady with the blue shirt'},
  {'tokens': ['lady', 'with', 'back', 'to', 'us'],
   'raw': 'lady w back to us',
   'sent_id': 1,
   'sent': 'lady with back to us'},
  {'tokens': ['blue', 'shirt'],
   'raw': 'blue shirt',
   'sent_id': 2,
   'sent': 'blue shirt'}],
 'category_id': 1}

In [83]:
class NewRefcocoDataset(torch.utils.data.Dataset):

    def __init__(self, refer, tokenizer, max_bb = 75):
        self.tokenizer = tokenizer
        self.refer = refer
        self.max_bb = max_bb
        self.sent_ids = self.get_sent_ids()
#         self.targets = labels
#         self.max_len = max_len

    def __len__(self):
        return len(self.train_ids)
    
    def get_sent_ids(self):
        sent_ids = []
        for ref_id in self.refer.getRefIds():
            ref = self.refer.Refs[ref_id]
            for sent_id in ref['sent_ids']:
                sent_ids.append(sent_id)
        return sent_ids

    def __getitem__(self, index):
        ref = refer.Refs[index]
        img_id = ref['image_id']
        ann_id = ref['ann_id']
        objs = refer.imgToAnns[img_id]
        obj_ids = [obj['id'] for obj in objs]
        
        sub_images = []
        for obj in objs:
            x_a = get_bounded_subimage(refer, img_id, obj['id'], xs=224,ys=224, show=False)
            if x_a is not None:
                sub_images.append(x_a)
        
        
        num_sub_images = len(sub_images)
        tokenized_sents = []
        text_masks = []
        text_labels = []
        text = []
        for sent in ref['sentences']:
            
            
            text_ids = tokenizer.encode(
                sent['sent'],
                padding="max_length",
                truncation=True,
                max_length=40,
                return_special_tokens_mask=True,
            )
            text_masks.append(torch.tensor([1 if text_ids[i]>0 else 0 for i,_ in enumerate(text_ids)]))
            text_labels.append(torch.tensor([-100 for i in range(40)]))
            text_ids = torch.tensor(text_ids)#.reshape(1,-1)
            tokenized_sents.append(text_ids)
            text.append(sent['sent'])
            
        ### TODO: Put all of the sub images in the infer dict with the coressponding sentence.
          
        return_dict = {
            'ann_id' : ann_id,
            'image' : sub_images,
            'obj_ids' : obj_ids,
            'text' : ref['sentences'],
            'text_ids' : tokenized_sents,
            'text_labels' : text_labels,
            'text_masks' : text_masks
        }


        return return_dict

In [84]:
ds = NewRefcocoDataset(refer, tokenizer)
# ds.sent_ids

In [85]:
ds = RefcocoDataset(refer, tokenizer)
ds

In [86]:
train_ds = torch.utils.data.Subset(ds, train_ids)
train_ds

In [87]:
train_params = {'batch_size': BATCH_SIZE,
                'shuffle': False,
                'num_workers': 0
                }

training_loader = torch.utils.data.DataLoader(ds, **train_params)

In [88]:
for i, data in enumerate(training_loader):
    if i==1:
        break
data

{'ann_id': tensor([463958]),
 'image': [tensor([[[[[0.0549, 0.0549, 0.0549,  ..., 0.0078, 0.0000, 0.0000],
             [0.0667, 0.0667, 0.0627,  ..., 0.0118, 0.0118, 0.0118],
             [0.0627, 0.0627, 0.0627,  ..., 0.0039, 0.0039, 0.0039],
             ...,
             [0.2824, 0.2588, 0.2157,  ..., 0.6784, 0.7176, 0.7412],
             [0.2745, 0.2549, 0.2235,  ..., 0.8588, 0.8627, 0.8627],
             [0.2745, 0.2588, 0.2353,  ..., 0.6941, 0.6745, 0.6627]],
  
            [[0.0510, 0.0510, 0.0510,  ..., 0.0118, 0.0078, 0.0039],
             [0.0667, 0.0667, 0.0627,  ..., 0.0118, 0.0118, 0.0118],
             [0.0627, 0.0627, 0.0627,  ..., 0.0039, 0.0000, 0.0000],
             ...,
             [0.4314, 0.3922, 0.3216,  ..., 0.7804, 0.8196, 0.8471],
             [0.4157, 0.3843, 0.3255,  ..., 0.8863, 0.8706, 0.8627],
             [0.4157, 0.3882, 0.3373,  ..., 0.6471, 0.6078, 0.5843]],
  
            [[0.0314, 0.0314, 0.0314,  ..., 0.0039, 0.0039, 0.0039],
             [0.0353,

### Training Loop with Data Class

In [94]:
model.train()
for data in tqdm(training_loader):
    obj_ids = data['obj_ids']
    ann_id = data['ann_id']
    for i,sent in enumerate(data['text']):
        scores = []
        optim.zero_grad()
        for sub_image in data['image']:
            ### TODO: Put all of the sub images in the infer dict with the coressponding sentence.
            input_dict = {
#                 'image' : [sub_image.squeeze(dim=0)],
                'image' : [sub_image.squeeze(dim=0)],
                'text' : sent['sent'],
                'text_ids' : data['text_ids'][i],
                'text_labels' : data['text_labels'][i],
                'text_masks' : data['text_masks'][i]
            }
            infer_dict = model.infer(input_dict)
            score = model.ref_classifier(infer_dict['cls_feats'])
            scores.append(score)
    
        target = torch.tensor([obj_ids.index(ann_id)])
        scores = torch.cat(scores)
        loss = loss_fn(scores.reshape(1,-1),target)
        loss.backward()

        # Adjust learning weights
        optim.step()

  0%|          | 0/42404 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [92]:
infer_dict['cls_feats'].shape

torch.Size([1, 384])

### Training Loop

In [102]:
# Create loop for ref res
train_ids = refer.getRefIds(split='train')
text_labels = [[-100 for i in range(40)]]
# train_ids = train_ids[:5]

gold = []
model.train()
for ref_id in tqdm(train_ids):
    ref = refer.Refs[ref_id]
    img_id = ref['image_id']
    ann_id = ref['ann_id']
    objs = refer.imgToAnns[img_id]
    obj_ids = [obj['id'] for obj in objs]
    
    sub_images = []
    for obj in objs:
        x_a = get_bounded_subimage(refer, img_id, obj['id'], xs=224,ys=224, show=False)
        if x_a is not None:
            sub_images.append(x_a)
    num_sub_images = len(sub_images)
        
    
    for sent in ref['sentences']:
        scores = []
        for sub_image in sub_images:
            text_ids = tokenizer.encode(
                sent['sent'],
                padding="max_length",
                truncation=True,
                max_length=40,
                return_special_tokens_mask=True,
            )
            text_masks = torch.tensor([1 if text_ids[i]>0 else 0 for i,_ in enumerate(text_ids)]).reshape(1,-1)
            optim.zero_grad()
            
            ### TODO: Put all of the sub images in the infer dict with the coressponding sentence.
          
            input_dict = {
                'image' : [sub_image],
                'text' : sent,
                'text_ids' : torch.tensor(text_ids).reshape(1,-1),
                'text_labels' : text_labels,
                'text_masks' : text_masks
            }
            infer_dict = model.infer(input_dict)
            score = model.ref_classifier(infer_dict['cls_feats'])
            scores.append(score)
        
        target = torch.tensor([obj_ids.index(ann_id)])
        scores = torch.cat(scores)
        loss = loss_fn(scores.reshape(1,-1),target)
        loss.backward()

        # Adjust learning weights
        optim.step()

  0%|          | 0/42404 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [107]:
input_dict = {
                'image' : sub_images,
                'text' : [sent for i in range(75-num_sub_images)],
                'text_ids' : torch.tensor(text_ids).reshape(1,-1),
                'text_labels' : text_labels,
                'text_masks' : text_masks
            }
infer_dict = model.infer(input_dict)
infer_dict['cls_feats']
score = model.ref_classifier(infer_dict['cls_feats'])
score

tensor([[ 2.3006,  2.6170,  1.6831,  1.2051,  1.8336, -0.1816,  0.0302,  0.0501,
         -2.1953, -1.9043, -1.4147, -1.7629, -2.0354, -1.6550, -2.3749, -2.3757,
          0.3474, -1.9056, -1.6654, -1.5946, -2.0714, -1.9997, -2.2295, -2.6123,
         -1.8829, -2.0418, -1.8176, -1.8748, -2.0110, -1.8707, -1.9237, -1.9641,
         -1.7356, -2.2223, -2.1576, -2.0927, -1.9364, -2.5002, -2.0571, -1.6524,
         -2.2496, -2.0270, -1.7219, -2.0512, -2.1569, -1.8583, -1.7638, -2.0023,
         -2.2118, -2.1786, -1.8194, -1.9417, -1.9718, -1.9179, -1.7175, -2.2702,
         -1.8537, -2.1222, -2.0437, -2.4136, -2.1960, -2.4836, -2.5931, -2.0901,
         -2.2981, -1.7321, -2.3820, -1.7381, -2.0016, -2.1951, -2.0354, -2.0756,
         -2.1454, -1.9025, -1.8638]], grad_fn=<AddmmBackward0>)

In [ ]:
infer_dict['cls_feats'].shape

In [ ]:
## Eval Loop
eval_ids = refer.getRefIds(split='val')
with torch.no_grad():
    gold = []
    for ref_id in tqdm(eval_ids):
        ref = refer.Refs[ref_id]
        img_id = ref['image_id']
        ann_id = ref['ann_id']
        objs = refer.imgToAnns[img_id]
        obj_ids = [obj['id'] for obj in objs]
    
        sub_images = []
        for obj in objs:
            x_a = get_bounded_subimage(refer, img_id, obj['id'], xs=224,ys=224, show=False)
            if x_a is not None:
                sub_images.append(x_a)
        num_sub_images = len(sub_images)
        for sent in ref['sentences']:
            scores = []
            text_ids = tokenizer.encode(
                sent['sent'],
                padding="max_length",
                truncation=True,
                max_length=40,
                return_special_tokens_mask=True,
            )
            text_masks = torch.tensor([1 if text_ids[i]>0 else 0 for i,_ in enumerate(text_ids)]).reshape(1,-1)
    
            for sub_image in sub_images:
                # assert not torch.isnan(x_a).any()
                input_dict = {
                    'image' : [sub_image],
                    'text' : sent,
                    'text_ids' : torch.tensor(text_ids).reshape(1,-1),
                    'text_labels' : text_labels,
                    'text_masks' : text_masks
                }
                infer_dict = model.infer(input_dict)
                score = model.ref_classifier(infer_dict['cls_feats'])
                scores.append(score)

            pred_index = np.argmax(scores)
            pred_id = objs[pred_index]['id']
            target = torch.tensor([obj_ids.index(ann_id)])
            scores = torch.cat(scores)
            if pred_id == ann_id:
                gold.append(1)
            else:
                gold.append(0)

## To Do:

Find maximum number of regions in a refcoco image, make this the classifier size



### Data Module

In [44]:
from refcoco_utils import _config
dm = MTDataModule(_config, dist=False)
dm.batch_size = 2

In [45]:
dm.prepare_data()
dm.setup('fit')
loader = dm.train_dataloader()

In [46]:
loader.batch_size

2

In [47]:
for i, batch in enumerate(loader):
    if i ==1:
        break
    
    

In [48]:
batch

{'replica': [True, True],
 'false_image_0': [tensor([[[[-1.6727, -1.6555, -1.6727,  ..., -2.0323, -1.9467, -1.6727],
            [-1.7240, -1.7069, -1.7069,  ..., -2.0323, -1.9124, -1.6898],
            [-1.5870, -1.5699, -1.5185,  ..., -2.0323, -1.8268, -1.7412],
            ...,
            [-1.6898, -1.6727, -1.6555,  ...,  1.7009,  1.7009,  1.6667],
            [-1.6384, -1.6384, -1.6384,  ...,  1.7180,  1.7180,  1.7009],
            [-1.5014, -1.5528, -1.5699,  ...,  1.7009,  1.7009,  1.6667]],
  
           [[-1.7031, -1.7031, -1.7031,  ..., -1.9307, -1.8782, -1.7031],
            [-1.6856, -1.6681, -1.6856,  ..., -1.9307, -1.8431, -1.7381],
            [-1.6506, -1.6681, -1.6331,  ..., -1.9482, -1.7731, -1.7906],
            ...,
            [-1.2829, -1.2829, -1.3004,  ...,  1.0280,  1.0280,  0.9930],
            [-1.2654, -1.2654, -1.2829,  ...,  0.9930,  0.9930,  0.9930],
            [-1.2304, -1.2304, -1.2304,  ...,  0.9755,  0.9755,  0.9405]],
  
           [[-1.5953, -1.56

In [52]:
infer_dict = model.infer(batch)
infer_dict['cls_feats'].shape

torch.Size([2, 384])

### Find Max Number of Objects

In [34]:
size = []
for ref_id in train_ids:
    ref = refer.Refs[ref_id]
    img_id = ref['image_id']
    ann_id = ref['ann_id']
    objs = refer.imgToAnns[img_id]
    size.append(len(objs))
max_size = max(size)
print(f'The most objects in any reference is {max_size}')


The most objects in any reference is 75


In [46]:
size = []
for ref_id in train_ids:
    ref = refer.Refs[ref_id]
    
    size.append(len(ref['sentences']))
max_size = max(size)
print(f'The most sentences in any reference is {max_size}')

The most sentences in any reference is 6


In [43]:
ref['sentences'].__len__()

2